# 10 · Open-Set Evaluation ("Not identified" and "Not skin")

**Purpose.** Tune and measure how well the system refuses photos it should not name: other skin diseases and photos that are not skin at all.

| | |
|---|---|
| **Reads** | `val_ensemble_probs.npy`, `test_ensemble_probs.npy`, `test_predictions.csv` |
| **Writes** | `reports/results/open_set.json`, figures |
| **Run time (Colab T4)** | ~3 min |
| **Next notebook** | 11 · Grad-CAM |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dermalens.data import make_loader, resolve_images
from dermalens.utils import get_device, load_json, save_json

info = load_json(paths.splits / "class_info.json")       # classes + weights frozen in notebook 05
CLASSES = info["classes"]                                  # 6 targets + "Other"
WEIGHTS = [info["class_weights"][name] for name in CLASSES]
train = pd.read_csv(paths.splits / "train.csv")
val = pd.read_csv(paths.splits / "val.csv")
IMAGE_ROOT, PATH_COL = resolve_images(train, paths)        # cache if present, else raw files
DEVICE = get_device()
IMAGE_SIZE = CONFIG["data"]["image_size"]
BATCH, WORKERS = CONFIG["train"]["batch_size"], CONFIG["train"]["num_workers"]
USE_META, COLOR = CONFIG["data"]["use_metadata"], CONFIG["data"]["color_constancy"]


def loader(rows, train_mode=False, size=IMAGE_SIZE, use_meta=USE_META, classes=None, label_col="label",
           seed=0):
    """One DataLoader with the project settings (augmentation + metadata dropout only when training)."""
    return make_loader(rows, IMAGE_ROOT, classes or CLASSES, size, train=train_mode, batch_size=BATCH,
                       num_workers=WORKERS, path_col=PATH_COL, label_col=label_col, use_metadata=use_meta,
                       meta_dropout=CONFIG["data"]["meta_dropout"], color_constancy=COLOR, seed=seed)


print(f"train {len(train)} | val {len(val)} | classes {len(CLASSES)} | device {DEVICE} | "
      f"images from {IMAGE_ROOT} ({PATH_COL})")

from dermalens import ood
from dermalens.labels import OTHER_CLASS

val_probs = np.load(paths.results / "val_ensemble_probs.npy")
val_true = np.load(paths.results / "val_true.npy")
test_probs = np.load(paths.results / "test_ensemble_probs.npy")
predictions = pd.read_csv(paths.results / "test_predictions.csv")
test_true = np.array([CLASSES.index(c) for c in predictions["true_label"]])
assert OTHER_CLASS in CLASSES, "Open-set evaluation needs data.open_set: true (notebooks 03 and 05)."
other = CLASSES.index(OTHER_CLASS)

## 1. How the app decides

1. **Skin check** - share of pixels in a skin-colour range (YCrCb). Below 15% → *not skin*.
2. **"Other" wins** → *not identified* (looks like a disease we do not cover).
3. **Confidence below a threshold** → *not identified* (too unsure).
4. Otherwise → the condition, or *healthy*.

The confidence threshold is set on **validation** so that 95% of known-class photos are still
answered; we then measure on test how many "Other" photos are correctly refused.

In [ ]:
known_columns = [i for i in range(len(CLASSES)) if i != other]


def known_confidence(probs):
    """Highest probability among the 6 known classes.

    ('Other' winning is handled separately by ood.decide. v3.0 set this score to 0 whenever
    'Other' won; since ~10% of known validation photos lose to 'Other', the 5% quantile became
    0.0 and the confidence check was switched off. Fixed in v3.1.)
    """
    return probs[:, known_columns].max(1)

val_known = val_true != other
THRESHOLD = ood.tune_threshold(known_confidence(val_probs)[val_known], CONFIG["open_set"]["target_tpr"])
print(f"Confidence threshold (keeps {CONFIG['open_set']['target_tpr']:.0%} of known val photos): {THRESHOLD:.3f}")

test_known = test_true != other
conf = known_confidence(test_probs)
auroc = ood.open_set_auroc(conf[test_known], conf[~test_known]) if (~test_known).any() else float("nan")
decisions = [ood.decide(p, CLASSES, THRESHOLD)[0] for p in test_probs]
predictions["status"] = decisions
rejected_other = float(np.mean([d == "not_identified" for d, k in zip(decisions, test_known) if not k]))
answered_known = float(np.mean([d != "not_identified" for d, k in zip(decisions, test_known) if k]))
correct_when_answered = float(predictions.loc[test_known & (predictions["status"] != "not_identified"),
                                              "correct"].mean())
print(f"Open-set AUROC (known vs other diseases): {auroc:.3f}")
print(f"Other-disease photos refused ('not identified'): {rejected_other:.1%}")
print(f"Known photos answered: {answered_known:.1%} | accuracy on answered: {correct_when_answered:.1%}")

## 2. Accuracy-coverage trade-off

Raising the threshold refuses more photos but makes the answers that remain more accurate. This
curve is a standard figure in selective-prediction papers: report accuracy at 100%, 90% and 80%
coverage.

In [ ]:
order = np.argsort(-conf[test_known])
correct_sorted = predictions.loc[test_known, "correct"].to_numpy()[order]
coverage = np.arange(1, len(order) + 1) / len(order)
accuracy_at = np.cumsum(correct_sorted) / np.arange(1, len(order) + 1)
plt.figure(figsize=(6, 4))
plt.plot(coverage, accuracy_at)
plt.xlabel("Coverage (share of known photos answered)")
plt.ylabel("Accuracy on answered photos")
plt.title("Selective prediction")
plt.grid(alpha=0.3)
plt.savefig(paths.figures / "accuracy_coverage.png", dpi=150, bbox_inches="tight")
plt.show()
selective = {f"accuracy_at_{int(c * 100)}pct_coverage": round(float(accuracy_at[max(int(c * len(order)) - 1, 0)]), 4)
             for c in (1.0, 0.9, 0.8, 0.7)}
print(selective)

## 3. Skin check on photos that are not skin

Put any non-skin photos (pets, objects, documents, screenshots) into `data/raw/non_skin/` to test with
real images. Generated pictures (plain colours, noise) are used in addition so the check always runs.

In [ ]:
from PIL import Image
from dermalens.data import load_rgb

rng = np.random.default_rng(0)
synthetic = [Image.new("RGB", (256, 256), tuple(int(v) for v in rng.integers(0, 255, 3))) for _ in range(30)]
synthetic += [Image.fromarray(rng.integers(0, 255, (256, 256, 3), dtype=np.uint8)) for _ in range(10)]
user_folder = paths.raw / "non_skin"
real_non_skin = [load_rgb(p) for p in sorted(user_folder.glob("*")) if p.suffix.lower() in (".jpg", ".jpeg", ".png")] \
    if user_folder.exists() else []
skin_rows = predictions.sample(min(200, len(predictions)), random_state=0)
skin_scores = [ood.skin_pixel_ratio(load_rgb(IMAGE_ROOT / p)) for p in skin_rows[PATH_COL]]
non_skin_scores = [ood.skin_pixel_ratio(im) for im in synthetic + real_non_skin]
limit = CONFIG["open_set"]["min_skin_ratio"]
skin_kept = float(np.mean(np.array(skin_scores) >= limit))
non_skin_refused = float(np.mean(np.array(non_skin_scores) < limit))
print(f"Real skin photos passing the check: {skin_kept:.1%} (want ~100%)")
print(f"Non-skin images refused: {non_skin_refused:.1%} ({len(real_non_skin)} real + {len(synthetic)} generated)")
plt.hist(skin_scores, bins=20, alpha=0.6, label="skin photos")
plt.hist(non_skin_scores, bins=20, alpha=0.6, label="non-skin")
plt.axvline(limit, color="k", linestyle="--")
plt.legend(); plt.xlabel("skin-pixel ratio"); plt.savefig(paths.figures / "skin_check.png", dpi=150); plt.show()

In [ ]:
save_json({"confidence_threshold": THRESHOLD, "open_set_auroc": auroc, "other_refused": rejected_other,
           "known_answered": answered_known, "accuracy_when_answered": correct_when_answered,
           **selective, "min_skin_ratio": limit, "skin_photos_kept": skin_kept,
           "non_skin_refused": non_skin_refused, "real_non_skin_images": len(real_non_skin)},
          paths.results / "open_set.json")
display(predictions["status"].value_counts().to_frame("test photos"))